# Chunking Strategies — Daily Essentials Policy Documents

Lecture 4, Part 5 named five chunking strategies: **fixed-size, paragraph-based,
sentence-based, section-based, and recursive splitting.** This notebook
implements all five against the actual `documents.py` used by `rag.py`, so we
can see — not just hear — how the choice of strategy changes what a retriever
has to work with.

Every strategy below returns the same shape: a list of
`{"text", "source", "strategy", "chunk_index", ...}` dicts, so they're
interchangeable with `rag.py`'s `build_index()`.

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## The source documents

Unchanged from `documents.py` — the same five mock policy PDFs `rag.py`
indexes, including the deliberate return-policy / warranty-policy split
behind the "mixer bought in the Diwali sale, stopped working after 20 days"
example.

In [2]:
POLICY_DOCUMENTS = [
    {
        "source": "return_policy.pdf",
        "text": """
General return policy: Most items purchased from Daily Essentials may be returned within 30 days of delivery for a full refund, provided the item is unused and in its original packaging.

Electronics: Electronics (including kitchen appliances such as mixers, blenders, and coffee makers) may be returned within 15 days of delivery, not 30. This shorter window applies whether or not the item is defective.

Clothing and textiles: Clothing, linens, and textile items may be returned within 30 days, and do not need to be in original packaging as long as tags are still attached.

Damaged or defective products: Items that arrive damaged, or that stop working after the standard return window has passed, are not eligible for a standard return — see warranty_policy.pdf for repair and replacement options instead.
""",
    },
    {
        "source": "refund_policy.pdf",
        "text": """
Refund method: Approved refunds are issued to the original payment method within 5-7 business days of Daily Essentials receiving the returned item.

Store credit option: Customers may choose store credit instead of a refund to the original payment method; store credit is issued immediately and never expires.

Shipping costs: Original shipping charges are non-refundable unless the return is due to a Daily Essentials error (wrong item shipped, item arrived damaged).
""",
    },
    {
        "source": "shipping_policy.pdf",
        "text": """
Standard shipping: Orders are normally delivered within 3-5 business days of being placed.

Festival sale shipping: During major sale events (Diwali Sale, End of Season Sale), delivery times may extend to 7-10 business days due to higher order volumes.

Tracking: A tracking number is emailed once an order ships, and can also be looked up in the Daily Essentials app under 'My Orders'.
""",
    },
    {
        "source": "warranty_policy.pdf",
        "text": """
Standard warranty: Electrical appliances, including mixers, blenders, and coffee makers, carry a one-year manufacturer's warranty against defects in materials and workmanship.

Warranty claims after delivery: If an appliance stops working within the one-year warranty period, the customer is entitled to a free repair or replacement, regardless of whether the standard return window (see return_policy.pdf) has already passed.

What voids the warranty: Warranty coverage does not apply to damage caused by misuse, unauthorized repairs, or normal wear on consumable parts (such as blender blades).

Festival sale items: Items purchased during a festival sale carry the same one-year warranty as items purchased at full price — the sale price does not shorten the warranty period.
""",
    },
    {
        "source": "exchange_policy.pdf",
        "text": """
Size and color exchanges: Clothing items can be exchanged for a different size or color within 30 days of delivery, subject to availability.

Exchanging electronics: Electronics are not eligible for a direct exchange; customers should follow the return process instead and place a new order.
""",
    },
]

print(f"Loaded {len(POLICY_DOCUMENTS)} documents")

Loaded 5 documents


## Strategy 1 — Fixed-size chunking (with overlap)

The bluntest approach: slice every `chunk_size` characters, sliding forward by
`chunk_size - overlap` each time so consecutive chunks share a strip of text.
Fast and simple, but it doesn't know or care where a sentence ends — it can
slice a sentence in half.

In [3]:
def chunk_fixed_size(documents, chunk_size=200, overlap=40):
    """Slide a fixed-width window across the text. `overlap` controls how
    much consecutive chunks share, so a sentence that straddles a boundary
    still appears whole in at least one chunk."""
    chunks = []
    for doc in documents:
        text = " ".join(doc["text"].split())  # collapse newlines/whitespace
        start, idx = 0, 0
        while start < len(text):
            end = start + chunk_size
            piece = text[start:end].strip()
            if piece:
                chunks.append({
                    "text": piece, "source": doc["source"],
                    "strategy": "fixed_size", "chunk_index": idx,
                })
                idx += 1
            if end >= len(text):
                break
            start += chunk_size - overlap
    return chunks


fixed_size_chunks = chunk_fixed_size(POLICY_DOCUMENTS)
print(f"{len(fixed_size_chunks)} chunks\n")
for c in fixed_size_chunks[:3]:
    print(f"[{c['source']} #{c['chunk_index']}] {c['text']!r}")

18 chunks

[return_policy.pdf #0] 'General return policy: Most items purchased from Daily Essentials may be returned within 30 days of delivery for a full refund, provided the item is unused and in its original packaging. Electronics:'
[return_policy.pdf #1] 'in its original packaging. Electronics: Electronics (including kitchen appliances such as mixers, blenders, and coffee makers) may be returned within 15 days of delivery, not 30. This shorter window a'
[return_policy.pdf #2] 'delivery, not 30. This shorter window applies whether or not the item is defective. Clothing and textiles: Clothing, linens, and textile items may be returned within 30 days, and do not need to be in'


### Seeing the overlap directly

Print the tail of chunk 0 and the head of chunk 1 side by side — the shared
text is the overlap doing its job (this is the "A B C D E F G / E F G H I J K"
picture from Lecture 4, just with real text).

In [4]:
return_policy_chunks = [c for c in fixed_size_chunks if c["source"] == "return_policy.pdf"]
print("End of chunk 0:  ", repr(return_policy_chunks[0]["text"][-55:]))
print("Start of chunk 1:", repr(return_policy_chunks[1]["text"][:55]))

End of chunk 0:   'm is unused and in its original packaging. Electronics:'
Start of chunk 1: 'in its original packaging. Electronics: Electronics (in'


## Strategy 2 — Paragraph-based chunking

Split on blank lines instead of a fixed character count — this is what
`rag.py`'s `chunk_documents()` currently does. It respects the document's own
structure (each policy point is already its own paragraph), so no sentence
ever gets cut mid-thought.

In [5]:
def chunk_paragraphs(documents):
    """Split on blank lines — each paragraph becomes one chunk."""
    chunks = []
    for doc in documents:
        paragraphs = [p.strip() for p in doc["text"].split("\n\n") if p.strip()]
        for i, p in enumerate(paragraphs):
            chunks.append({
                "text": p, "source": doc["source"],
                "strategy": "paragraph", "chunk_index": i,
            })
    return chunks


paragraph_chunks = chunk_paragraphs(POLICY_DOCUMENTS)
print(f"{len(paragraph_chunks)} chunks\n")
for c in paragraph_chunks[:3]:
    print(f"[{c['source']} #{c['chunk_index']}] {c['text'][:80]!r}...")

16 chunks

[return_policy.pdf #0] 'General return policy: Most items purchased from Daily Essentials may be returne'...
[return_policy.pdf #1] 'Electronics: Electronics (including kitchen appliances such as mixers, blenders,'...
[return_policy.pdf #2] 'Clothing and textiles: Clothing, linens, and textile items may be returned withi'...


## Strategy 3 — Sentence-based chunking

Split on sentence boundaries, then group a few sentences per chunk (grouping
size is tunable — one sentence alone is often too little context to retrieve
usefully). Notice this can merge text that paragraph-based chunking would
have kept separate, since it ignores blank lines entirely.

In [6]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def chunk_sentences(documents, sentences_per_chunk=2):
    """Split into sentences, then group N sentences per chunk."""
    chunks = []
    for doc in documents:
        text = " ".join(doc["text"].split())
        sentences = [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]
        for i in range(0, len(sentences), sentences_per_chunk):
            group = sentences[i:i + sentences_per_chunk]
            chunks.append({
                "text": " ".join(group), "source": doc["source"],
                "strategy": "sentence", "chunk_index": i // sentences_per_chunk,
            })
    return chunks


sentence_chunks = chunk_sentences(POLICY_DOCUMENTS)
print(f"{len(sentence_chunks)} chunks\n")
for c in sentence_chunks[:3]:
    print(f"[{c['source']} #{c['chunk_index']}] {c['text'][:90]!r}...")

10 chunks

[return_policy.pdf #0] 'General return policy: Most items purchased from Daily Essentials may be returned within 3'...
[return_policy.pdf #1] 'This shorter window applies whether or not the item is defective. Clothing and textiles: C'...
[return_policy.pdf #2] 'Damaged or defective products: Items that arrive damaged, or that stop working after the s'...


## Strategy 4 — Section-based chunking

Every paragraph in these policy docs already starts with a label — `"Electronics:"`,
`"Standard warranty:"`, `"Festival sale shipping:"` — the document's own section
headers. This strategy splits on *those markers specifically*, not on blank
lines. It happens to land on the same boundaries as paragraph-based chunking
here (because these mock docs are already well-formatted), but the mechanism
is different: it would keep working even if two sections got mashed onto the
same line with no blank line between them, which pure blank-line splitting
would miss. It also gives us a real section **title** for metadata, useful
for citations.

In [7]:
SECTION_HEADER_RE = re.compile(r"(?m)^([A-Z][A-Za-z0-9 /\-]{2,40}):\s")


def chunk_sections(documents):
    """Split on 'Label:' style section headers found at the start of a line,
    rather than on blank lines. Keeps the header text as metadata."""
    chunks = []
    for doc in documents:
        text = doc["text"].strip()
        matches = list(SECTION_HEADER_RE.finditer(text))
        if not matches:
            chunks.append({
                "text": text, "source": doc["source"],
                "strategy": "section", "chunk_index": 0, "section": None,
            })
            continue
        for i, m in enumerate(matches):
            start = m.start()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            section_text = " ".join(text[start:end].strip().split())
            chunks.append({
                "text": section_text, "source": doc["source"], "strategy": "section",
                "chunk_index": i, "section": m.group(1),
            })
    return chunks


section_chunks = chunk_sections(POLICY_DOCUMENTS)
print(f"{len(section_chunks)} chunks\n")
for c in section_chunks[:4]:
    print(f"[{c['source']}] section={c['section']!r} -> {c['text'][:60]!r}...")

16 chunks

[return_policy.pdf] section='General return policy' -> 'General return policy: Most items purchased from Daily Essen'...
[return_policy.pdf] section='Electronics' -> 'Electronics: Electronics (including kitchen appliances such '...
[return_policy.pdf] section='Clothing and textiles' -> 'Clothing and textiles: Clothing, linens, and textile items m'...
[return_policy.pdf] section='Damaged or defective products' -> 'Damaged or defective products: Items that arrive damaged, or'...


## Strategy 5 — Recursive splitting

The most common real-world default (this is the idea behind LangChain's
`RecursiveCharacterTextSplitter`). Try the *largest* meaningful unit first —
a paragraph. If it already fits under `max_chars`, keep it whole. If it's too
big, fall back to sentences and greedily pack them until the limit. If even a
single sentence is too big, fall back to a hard fixed-size slice as a last
resort. It's a hierarchy of the other four strategies, applied in order of
preference.

In [8]:
def chunk_recursive(documents, max_chars=200, overlap=30):
    """Prefer whole paragraphs; fall back to sentence-packing, then to a
    hard fixed-size slice, only when a unit is too big to keep whole."""
    chunks = []
    for doc in documents:
        paragraphs = [p.strip() for p in doc["text"].split("\n\n") if p.strip()]
        idx = 0
        for p in paragraphs:
            p = " ".join(p.split())
            if len(p) <= max_chars:
                chunks.append({
                    "text": p, "source": doc["source"], "strategy": "recursive",
                    "chunk_index": idx, "split_level": "paragraph",
                })
                idx += 1
                continue

            # paragraph too big -> fall back to packing sentences
            sentences = [s.strip() for s in SENTENCE_SPLIT_RE.split(p) if s.strip()]
            buffer = ""
            for s in sentences:
                candidate = (buffer + " " + s).strip() if buffer else s
                if len(candidate) <= max_chars:
                    buffer = candidate
                    continue
                if buffer:
                    chunks.append({
                        "text": buffer, "source": doc["source"], "strategy": "recursive",
                        "chunk_index": idx, "split_level": "sentence",
                    })
                    idx += 1
                    buffer = ""
                if len(s) <= max_chars:
                    buffer = s
                else:
                    # even one sentence is too big -> hard fixed-size fallback
                    start = 0
                    while start < len(s):
                        piece = s[start:start + max_chars].strip()
                        chunks.append({
                            "text": piece, "source": doc["source"], "strategy": "recursive",
                            "chunk_index": idx, "split_level": "fixed_size_fallback",
                        })
                        idx += 1
                        start += max_chars - overlap
                    buffer = ""
            if buffer:
                chunks.append({
                    "text": buffer, "source": doc["source"], "strategy": "recursive",
                    "chunk_index": idx, "split_level": "sentence",
                })
                idx += 1
    return chunks


recursive_chunks = chunk_recursive(POLICY_DOCUMENTS)
print(f"{len(recursive_chunks)} chunks\n")

# Which fallback level did each chunk actually need?
from collections import Counter
level_counts = Counter(c["split_level"] for c in recursive_chunks)
print("Split levels used:", dict(level_counts))

19 chunks

Split levels used: {'paragraph': 13, 'sentence': 2, 'fixed_size_fallback': 4}


## Comparing all five strategies

Same five documents, five very different chunk sets. More chunks generally
means smaller, more precise units (good for pinpoint retrieval, worse for
preserving context); fewer, larger chunks mean the opposite trade-off.

In [9]:
strategies = {
    "fixed_size": fixed_size_chunks,
    "paragraph": paragraph_chunks,
    "sentence": sentence_chunks,
    "section": section_chunks,
    "recursive": recursive_chunks,
}

rows = []
for name, chunk_list in strategies.items():
    lengths = [len(c["text"]) for c in chunk_list]
    rows.append({
        "strategy": name,
        "num_chunks": len(chunk_list),
        "avg_chars": round(sum(lengths) / len(lengths), 1),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
    })

comparison_df = pd.DataFrame(rows).set_index("strategy")
comparison_df

,num_chunks,avg_chars,min_chars,max_chars
strategy,,,,
fixed_size,18,179.8,64,200
paragraph,16,169.5,90,249
sentence,10,271.8,132,425
section,16,169.5,90,249
recursive,19,145.7,62,200


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

colors = ["#2F6FED", "#12B886", "#E8890C", "#5C6BC0", "#1B2340"]

axes[0].bar(comparison_df.index, comparison_df["num_chunks"], color=colors)
axes[0].set_title("Number of Chunks by Strategy", fontweight="bold")
axes[0].set_ylabel("chunk count")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(comparison_df.index, comparison_df["avg_chars"], color=colors)
axes[1].set_title("Average Chunk Size by Strategy", fontweight="bold")
axes[1].set_ylabel("avg characters")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Does the strategy change what gets retrieved?

This is the part that actually matters for RAG. We'll build a tiny index for
*each* strategy's chunk set (same embedding model `rag.py` uses) and run the
Lecture 4 example query against all five — watch whether the **top result**
changes depending on how the documents were cut up.

*(Requires `sentence-transformers`; skip this cell if it isn't installed —
everything above runs without it.)*

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

query = "I bought a mixer during the Diwali sale and it stopped working after 20 days"
query_vec = model.encode([query], normalize_embeddings=True)[0]

for name, chunk_list in strategies.items():
    texts = [c["text"] for c in chunk_list]
    vecs = model.encode(texts, normalize_embeddings=True)
    scores = vecs @ query_vec
    top_i = int(np.argmax(scores))
    top_chunk = chunk_list[top_i]
    print(f"[{name:10s}] top match ({scores[top_i]:.3f}) from {top_chunk['source']}:")
    print(f"             {top_chunk['text'][:140]!r}...\n")

## Takeaways

- **Fixed-size** is the simplest to implement and the easiest to get wrong —
  it can slice a key sentence in half with no regard for meaning.
- **Paragraph-based** (what `rag.py` uses today) works well *because* these
  mock policy docs are already written as one clean idea per paragraph — it
  would work far worse on a wall-of-text PDF with no blank lines.
- **Sentence-based** trades context for precision — smaller, more surgical
  units, at the risk of losing the surrounding sentence that explains *why*.
- **Section-based** is the most metadata-rich (real section titles for
  citations) but depends on the source having identifiable headers at all.
- **Recursive** is the most robust default precisely because it *isn't* one
  strategy — it tries the better-structured option first and only degrades
  to a hard slice when it has no other choice.

None of this is free of trade-offs — exactly Lecture 4 Part 8's point about
RAG in general: the strategy you pick changes what the retriever *can* find,
but picking one doesn't guarantee it finds the *right* thing.